# M6.6 — False Positive / False Negative error analysis

Milestone:

`M6 — Evaluation + Error Analysis`

Substep:

`M6.6 — False Positive / False Negative Error Analysis`

Work type:

`RUNTIME LINEAGE RECONSTRUCTION / TRANSACTION-LEVEL ERROR ANALYSIS`

Câu hỏi trung tâm:

> Fraud bị bỏ sót và false alert có pattern gì trên canonical validation population?

Primary groups:

```text
Fraud side:
FN vs TP

Non-fraud side:
FP vs TN
```

Candidate semantic dimensions được phân tích:

```text
amount_numeric
transaction_mode
location_state
hour_of_day
day_of_week
is_new_merchant
has_prior_card_history
time_since_previous_transaction_min
transactions_last_1h
amount_minus_previous_mean
```

M6.6 còn đọc risk score của FN/FP để phân biệt:

- error gần baseline decision boundary;
- error có score rất xa boundary;
- model-specific / shared error subsets.

M6.6 không:

- retrain model;
- tuning;
- resampling;
- threshold optimization;
- calibration tuning;
- final model selection;
- FINAL TEST evaluation.

Runtime-dependent status trước khi chạy:

`NOT YET VERIFIED`

## 1. Lineage requirement trước mọi row-level finding

M6.2 mới xác minh:

```text
row_id_validation:
valid canonical lineage anchor
```

nhưng chưa chứng minh transaction-level semantic remapping.

M6.6 phải khóa chuỗi sau:

```text
M4.7 row_id_validation
        ↓ exact equality
raw artifact raw_row_id
        ↓
M4.7 deterministic representation
        ↓
strict-causal behavioral features
        ↓
canonical validation semantic frame
        ↓ exact row alignment
y_validation / y_pred / risk_score
        ↓
TP / FP / FN / TN transaction groups
```

Nếu row-id mapping, target alignment hoặc semantic reconstruction fail:

`STOP — KHÔNG ĐƯỢC INTERPRET ERROR PATTERN`

## 2. Target-leakage / FINAL TEST guardrail

M6.6 rebuild semantic features từ raw artifact nhưng:

```text
Is Fraud?
NOT READ

Errors?
NOT READ
```

Raw target không được dùng để tạo:

- behavioral history;
- semantic features;
- error groups.

Error-analysis label đến từ:

`M4.7 canonical y_validation.npy`

sau khi exact `row_id_validation` alignment PASS.

Historical feature state chỉ dùng transaction có:

`Timestamp < 2019-06-01`

và strict history:

`Timestamp(history) < Timestamp(current)`.

Các row từ `2019-06-01` trở đi không được đưa vào history state, semantic validation frame hoặc error-analysis statistics.

## 3. Probability context boundary

Persisted risk score:

`positive-class predict_proba`

M6.6 dùng score để mô tả error behavior.

Reference boundary:

`0.5`

chỉ được dùng sau khi runtime xác minh persisted `y_pred` tương thích với score-side của baseline default decision rule.

Predeclared descriptive bands:

```text
FN:
very_low_score      < 0.10
low_score           0.10 → < 0.40
near_boundary       0.40 → 0.50

FP:
near_boundary       0.50 → < 0.60
high_score          0.60 → < 0.90
very_high_score     >= 0.90
```

Các band này:

`DESCRIPTIVE ONLY`

Không phải threshold search và không được dùng để thay `y_pred`.

In [1]:

from pathlib import Path
from collections import Counter
import hashlib
import json
import platform
import sys
import time

import numpy as np
import pandas as pd


print("Python:")
print(sys.version)

print("\nExecutable:")
print(sys.executable)

print("\nPlatform:")
print(platform.platform())

print("\nNumPy:")
print(np.__version__)

print("\npandas:")
print(pd.__version__)


Python:
3.14.6 (main, Jun 10 2026, 10:03:53) [Clang 21.0.0 (clang-2100.0.123.102)]

Executable:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/.venv/bin/python

Platform:
macOS-26.6.2-arm64-arm-64bit-Mach-O

NumPy:
2.5.3

pandas:
3.0.5


## 4. Locate canonical raw / M4 / M6 artifacts

In [2]:

RAW_REL = (
    Path("data")
    / "raw"
    / "ibm_tabformer"
    / "card_transaction.v1.csv"
)

M4_REL = (
    Path("data")
    / "processed"
    / "m4_07_baseline_ready"
)

M6_02_REL = (
    Path("data")
    / "processed"
    / "m6_02_evaluation_artifact_audit"
)

M6_05_REL = (
    Path("data")
    / "processed"
    / "m6_05_cross_model_comparative_evaluation"
)

M6_06_REL = (
    Path("data")
    / "processed"
    / "m6_06_fp_fn_error_analysis"
)


candidate_roots = [
    Path.cwd(),
    *list(Path.cwd().parents)[:6],
]


PROJECT_ROOT = None

required_rel_paths = [
    RAW_REL,
    M4_REL / "y_validation.npy",
    M4_REL / "row_id_validation.npy",
    M6_02_REL / "m6_02_evaluation_registry.json",
    M6_05_REL / "m6_05_cross_model_comparison.json",
    M6_05_REL / "m6_05_comparison_manifest.json",
]


for candidate in candidate_roots:
    candidate = candidate.resolve()

    if all(
        (
            candidate
            / rel_path
        ).exists()
        for rel_path
        in required_rel_paths
    ):
        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy PROJECT_ROOT chứa đầy đủ "
        "raw artifact + M4.7 + M6.2 + M6.5 artifacts."
    )


RAW_PATH = PROJECT_ROOT / RAW_REL
M4_DIR = PROJECT_ROOT / M4_REL
M6_02_DIR = PROJECT_ROOT / M6_02_REL
M6_05_DIR = PROJECT_ROOT / M6_05_REL
OUTPUT_DIR = PROJECT_ROOT / M6_06_REL

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


REGISTRY_PATH = (
    M6_02_DIR
    / "m6_02_evaluation_registry.json"
)

M6_05_COMPARISON_PATH = (
    M6_05_DIR
    / "m6_05_cross_model_comparison.json"
)

M6_05_MANIFEST_PATH = (
    M6_05_DIR
    / "m6_05_comparison_manifest.json"
)


EXPECTED_RAW_FILE_SIZE = 2_354_626_737
EXPECTED_RAW_ROWS = 24_386_900
EXPECTED_CARD_COUNT = 6_139
EXPECTED_CONTEXT_ROWS = 23_038_920
EXPECTED_VALIDATION_ROWS = 712_458
EXPECTED_VALIDATION_FRAUD = 1_052

CHUNK_SIZE = 500_000

TRAIN_END = pd.Timestamp(
    "2019-01-01"
)

VALIDATION_END = pd.Timestamp(
    "2019-06-01"
)


assert RAW_PATH.exists()
assert (
    RAW_PATH.stat().st_size
    == EXPECTED_RAW_FILE_SIZE
)


print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nRAW_PATH:")
print(RAW_PATH)

print("\nOUTPUT_DIR:")
print(OUTPUT_DIR)

print(
    "\nM6.6 ENVIRONMENT / RAW ARTIFACT GATE: PASS"
)


PROJECT_ROOT:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening

RAW_PATH:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/raw/ibm_tabformer/card_transaction.v1.csv

OUTPUT_DIR:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m6_06_fp_fn_error_analysis

M6.6 ENVIRONMENT / RAW ARTIFACT GATE: PASS


## 5. Verify upstream evaluation handoff

In [3]:

with open(
    REGISTRY_PATH,
    "r",
    encoding="utf-8",
) as file:
    m6_02_registry = json.load(
        file
    )


with open(
    M6_05_COMPARISON_PATH,
    "r",
    encoding="utf-8",
) as file:
    m6_05_comparison = json.load(
        file
    )


with open(
    M6_05_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    m6_05_manifest = json.load(
        file
    )


assert (
    m6_02_registry[
        "official_run_count"
    ]
    == 6
)

assert (
    m6_02_registry[
        "validation_rows"
    ]
    == EXPECTED_VALIDATION_ROWS
)

assert (
    m6_02_registry[
        "validation_fraud_rows"
    ]
    == EXPECTED_VALIDATION_FRAUD
)

assert (
    m6_02_registry[
        "final_test_accessed"
    ]
    is False
)


assert (
    m6_05_comparison[
        "validation_rows"
    ]
    == EXPECTED_VALIDATION_ROWS
)

assert (
    m6_05_comparison[
        "validation_fraud_rows"
    ]
    == EXPECTED_VALIDATION_FRAUD
)

assert (
    m6_05_comparison[
        "selection_state"
    ][
        "final_model"
    ]
    == "OPEN"
)

assert (
    m6_05_comparison[
        "selection_state"
    ][
        "final_threshold"
    ]
    == "OPEN"
)

assert (
    m6_05_comparison[
        "final_test_accessed"
    ]
    is False
)


assert (
    m6_05_manifest[
        "probability_artifacts_verified"
    ]
    == 6
)

assert (
    m6_05_manifest[
        "threshold_search"
    ]
    == "NOT_PERFORMED"
)

assert (
    m6_05_manifest[
        "final_model_selected"
    ]
    is False
)

assert (
    m6_05_manifest[
        "final_test_accessed"
    ]
    is False
)


print(
    "M6.6 UPSTREAM HANDOFF GATE: PASS"
)


M6.6 UPSTREAM HANDOFF GATE: PASS


## 6. Load canonical validation target / row lineage

In [4]:

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


Y_VALIDATION_PATH = (
    M4_DIR
    / "y_validation.npy"
)

ROW_ID_VALIDATION_PATH = (
    M4_DIR
    / "row_id_validation.npy"
)


y_validation = np.load(
    Y_VALIDATION_PATH,
    allow_pickle=False,
)

row_id_validation = np.load(
    ROW_ID_VALIDATION_PATH,
    allow_pickle=False,
)


assert y_validation.ndim == 1
assert row_id_validation.ndim == 1

assert len(
    y_validation
) == EXPECTED_VALIDATION_ROWS

assert len(
    row_id_validation
) == EXPECTED_VALIDATION_ROWS

assert int(
    y_validation.sum()
) == EXPECTED_VALIDATION_FRAUD

assert len(
    np.unique(
        row_id_validation
    )
) == EXPECTED_VALIDATION_ROWS


assert (
    sha256_file(
        Y_VALIDATION_PATH
    )
    == m6_02_registry[
        "y_validation_sha256"
    ]
)

assert (
    sha256_file(
        ROW_ID_VALIDATION_PATH
    )
    == m6_02_registry[
        "row_id_validation_sha256"
    ]
)


print(
    "y_validation SHA256:"
)
print(
    m6_02_registry[
        "y_validation_sha256"
    ]
)

print(
    "\nrow_id_validation SHA256:"
)
print(
    m6_02_registry[
        "row_id_validation_sha256"
    ]
)

print(
    "\nM6.6 CANONICAL VALIDATION LINEAGE GATE: PASS"
)


y_validation SHA256:
0a21b2e93e017eda8b794be1a94a692beebc5883c1303427bc245299b55f5306

row_id_validation SHA256:
e7f3074c8604680eccef60ff7be45501f0dc10efbf6ac82e47bcbd1b00e43adc

M6.6 CANONICAL VALIDATION LINEAGE GATE: PASS


## 7. Exact M4.7 deterministic representation helpers

Các helper dưới đây giữ semantic của M4.7:

```text
Amount:
strip "$" / "," → numeric

Timestamp:
Year + Month + Day + Time

transaction_mode:
normalized Use Chip

location_state:
NON_PHYSICAL_OR_ONLINE
PHYSICAL_COMPLETE
PHYSICAL_ZIP_UNAVAILABLE

hour_of_day:
Timestamp hour

day_of_week:
Timestamp dayofweek
```

Không casefold location text ngoài contract đã khóa.

In [5]:

def parse_amount(
    amount_series,
):
    cleaned = (
        amount_series
        .astype("string")
        .str.replace(
            "$",
            "",
            regex=False,
        )
        .str.replace(
            ",",
            "",
            regex=False,
        )
        .str.strip()
    )

    return pd.to_numeric(
        cleaned,
        errors="coerce",
    )


def build_timestamp(
    df,
):
    date_part = pd.to_datetime(
        {
            "year":
                df["Year"],

            "month":
                df["Month"],

            "day":
                df["Day"],
        },
        errors="coerce",
    )

    time_part = pd.to_timedelta(
        (
            df["Time"]
            .astype("string")
            + ":00"
        ),
        errors="coerce",
    )

    return (
        date_part
        + time_part
    )


def normalize_string(
    series,
):
    return (
        series
        .astype("string")
        .str.strip()
    )


def assign_location_state(
    df,
):
    city = normalize_string(
        df[
            "Merchant City"
        ]
    )

    city_online = (
        city
        .eq("ONLINE")
        .fillna(False)
    )

    state_missing = (
        df[
            "Merchant State"
        ]
        .isna()
    )

    zip_missing = (
        df["Zip"]
        .isna()
    )

    nonphysical = (
        city_online
        & state_missing
        & zip_missing
    )

    physical_complete = (
        (~city_online)
        & (~state_missing)
        & (~zip_missing)
    )

    physical_zip_unavailable = (
        (~city_online)
        & (~state_missing)
        & zip_missing
    )

    result = np.select(
        [
            nonphysical,
            physical_complete,
            physical_zip_unavailable,
        ],
        [
            "NON_PHYSICAL_OR_ONLINE",
            "PHYSICAL_COMPLETE",
            "PHYSICAL_ZIP_UNAVAILABLE",
        ],
        default="OTHER_INCONSISTENT",
    )

    return pd.Series(
        result,
        index=df.index,
        dtype="string",
    )


print(
    "M6.6 DETERMINISTIC REPRESENTATION HELPERS: DEFINED"
)


M6.6 DETERMINISTIC REPRESENTATION HELPERS: DEFINED


## 8. Card-block streaming helper

`raw_row_id` được tái tạo đúng như M4.7:

```text
zero-based physical data-row offset
```

Raw usecols cố ý không chứa:

```text
Is Fraud?
Errors?
```

In [6]:

SEMANTIC_USECOLS = [
    "User",
    "Card",
    "Year",
    "Month",
    "Day",
    "Time",
    "Amount",
    "Use Chip",
    "Merchant Name",
    "Merchant City",
    "Merchant State",
    "Zip",
]


assert (
    "Is Fraud?"
    not in SEMANTIC_USECOLS
)

assert (
    "Errors?"
    not in SEMANTIC_USECOLS
)


def iter_card_blocks(
    data_path,
    chunksize,
    usecols,
):
    pending_key = None
    pending_parts = []

    raw_row_offset = 0

    for chunk in pd.read_csv(
        data_path,
        usecols=usecols,
        chunksize=chunksize,
    ):
        chunk_length = len(
            chunk
        )

        chunk[
            "raw_row_id"
        ] = np.arange(
            raw_row_offset,
            raw_row_offset
            + chunk_length,
            dtype=np.int64,
        )

        raw_row_offset += (
            chunk_length
        )

        chunk[
            "Timestamp"
        ] = build_timestamp(
            chunk
        )

        chunk[
            "Amount_numeric"
        ] = parse_amount(
            chunk[
                "Amount"
            ]
        )

        if (
            chunk[
                "Timestamp"
            ]
            .isna()
            .any()
        ):
            raise ValueError(
                "Timestamp parse failure."
            )

        if (
            chunk[
                "Amount_numeric"
            ]
            .isna()
            .any()
        ):
            raise ValueError(
                "Amount parse failure."
            )

        if (
            chunk[
                "Merchant Name"
            ]
            .isna()
            .any()
        ):
            raise ValueError(
                "Merchant Name missing."
            )

        working_columns = [
            "raw_row_id",
            "User",
            "Card",
            "Timestamp",
            "Amount_numeric",
            "Use Chip",
            "Merchant Name",
            "Merchant City",
            "Merchant State",
            "Zip",
        ]

        working = (
            chunk[
                working_columns
            ]
        )

        users = (
            working[
                "User"
            ]
            .to_numpy()
        )

        cards = (
            working[
                "Card"
            ]
            .to_numpy()
        )

        change_positions = (
            np.flatnonzero(
                (
                    users[1:]
                    != users[:-1]
                )
                |
                (
                    cards[1:]
                    != cards[:-1]
                )
            )
            + 1
        )

        starts = np.concatenate(
            [
                [0],
                change_positions,
            ]
        )

        ends = np.concatenate(
            [
                change_positions,
                [len(working)],
            ]
        )

        for start, end in zip(
            starts,
            ends,
        ):
            key = (
                int(
                    users[start]
                ),
                int(
                    cards[start]
                ),
            )

            segment = (
                working
                .iloc[
                    start:end
                ]
                .copy()
            )

            if pending_key is None:
                pending_key = key
                pending_parts = [
                    segment
                ]
                continue

            if key == pending_key:
                pending_parts.append(
                    segment
                )
                continue

            if (
                len(
                    pending_parts
                )
                == 1
            ):
                card_block = (
                    pending_parts[0]
                    .reset_index(
                        drop=True
                    )
                )
            else:
                card_block = (
                    pd.concat(
                        pending_parts,
                        ignore_index=True,
                    )
                )

            yield (
                pending_key,
                card_block,
            )

            pending_key = key
            pending_parts = [
                segment
            ]

    if (
        pending_key
        is not None
    ):
        if (
            len(
                pending_parts
            )
            == 1
        ):
            card_block = (
                pending_parts[0]
                .reset_index(
                    drop=True
                )
            )
        else:
            card_block = (
                pd.concat(
                    pending_parts,
                    ignore_index=True,
                )
            )

        yield (
            pending_key,
            card_block,
        )


print(
    "M6.6 CARD-BLOCK STREAMING HELPER: DEFINED"
)


M6.6 CARD-BLOCK STREAMING HELPER: DEFINED


## 9. Exact strict-causal behavioral builder

Core behavioral features:

```text
time_since_previous_transaction_min
transactions_last_1h
amount_minus_previous_mean
is_new_merchant
has_prior_card_history
```

Same-timestamp transaction:

`DO NOT PROVIDE HISTORY TO EACH OTHER`

In [7]:

ONE_HOUR_NS = int(
    pd.Timedelta(
        hours=1
    ).value
)


def compute_causal_behavioral_features(
    card_df,
):
    timestamp_ns = (
        card_df[
            "Timestamp"
        ]
        .to_numpy(
            dtype="datetime64[ns]"
        )
        .astype(
            "int64"
        )
    )

    if (
        np.diff(
            timestamp_ns
        )
        < 0
    ).any():
        raise ValueError(
            "Timestamp giảm bên trong Card block."
        )

    group_starts = np.concatenate(
        [
            [0],
            (
                np.flatnonzero(
                    timestamp_ns[1:]
                    != timestamp_ns[:-1]
                )
                + 1
            ),
        ]
    )

    group_ends = np.concatenate(
        [
            group_starts[1:],
            [len(card_df)],
        ]
    )

    group_lengths = (
        group_ends
        - group_starts
    )

    group_timestamp_ns = (
        timestamp_ns[
            group_starts
        ]
    )

    prior_count = np.repeat(
        group_starts,
        group_lengths,
    )

    has_prior_card_history = (
        prior_count
        > 0
    )

    time_since_group = np.full(
        len(
            group_starts
        ),
        np.nan,
        dtype="float64",
    )

    if (
        len(
            group_starts
        )
        > 1
    ):
        time_since_group[
            1:
        ] = (
            (
                group_timestamp_ns[
                    1:
                ]
                -
                group_timestamp_ns[
                    :-1
                ]
            )
            / 60_000_000_000
        )

    time_since_previous_min = (
        np.repeat(
            time_since_group,
            group_lengths,
        )
    )

    left_1h = np.searchsorted(
        timestamp_ns,
        (
            group_timestamp_ns
            - ONE_HOUR_NS
        ),
        side="left",
    )

    transactions_last_1h_group = (
        group_starts
        - left_1h
    )

    transactions_last_1h = (
        np.repeat(
            transactions_last_1h_group,
            group_lengths,
        )
    )

    amount = (
        card_df[
            "Amount_numeric"
        ]
        .to_numpy(
            dtype="float64"
        )
    )

    amount_prefix_sum = np.concatenate(
        [
            [0.0],
            np.cumsum(
                amount
            ),
        ]
    )

    previous_amount_mean_group = np.full(
        len(
            group_starts
        ),
        np.nan,
        dtype="float64",
    )

    has_prior_group = (
        group_starts
        > 0
    )

    previous_amount_mean_group[
        has_prior_group
    ] = (
        amount_prefix_sum[
            group_starts[
                has_prior_group
            ]
        ]
        /
        group_starts[
            has_prior_group
        ]
    )

    previous_amount_mean = np.repeat(
        previous_amount_mean_group,
        group_lengths,
    )

    amount_minus_previous_mean = (
        amount
        - previous_amount_mean
    )

    merchant_values = (
        card_df[
            "Merchant Name"
        ]
        .to_numpy()
    )

    is_new_merchant = np.empty(
        len(
            card_df
        ),
        dtype=bool,
    )

    seen_merchants = set()

    for start, end in zip(
        group_starts,
        group_ends,
    ):
        current_group_merchants = (
            merchant_values[
                start:end
            ]
        )

        is_new_merchant[
            start:end
        ] = np.array(
            [
                merchant
                not in seen_merchants
                for merchant
                in current_group_merchants
            ],
            dtype=bool,
        )

        seen_merchants.update(
            current_group_merchants
            .tolist()
        )

    return pd.DataFrame(
        {
            "raw_row_id":
                card_df[
                    "raw_row_id"
                ].to_numpy(),

            "Timestamp":
                card_df[
                    "Timestamp"
                ].to_numpy(),

            "amount_numeric":
                amount,

            "time_since_previous_transaction_min":
                time_since_previous_min,

            "transactions_last_1h":
                transactions_last_1h,

            "amount_minus_previous_mean":
                amount_minus_previous_mean,

            "is_new_merchant":
                is_new_merchant,

            "has_prior_card_history":
                has_prior_card_history,
        }
    )


print(
    "M6.6 STRICT-CAUSAL BEHAVIORAL BUILDER: DEFINED"
)


M6.6 STRICT-CAUSAL BEHAVIORAL BUILDER: DEFINED


## 10. Strict-causal regression check

In [8]:

causal_test_df = pd.DataFrame(
    {
        "raw_row_id":
            [0, 1, 2, 3],

        "Timestamp":
            pd.to_datetime(
                [
                    "2018-01-01 09:00:00",
                    "2018-01-01 10:00:00",
                    "2018-01-01 10:00:00",
                    "2018-01-01 10:30:00",
                ]
            ),

        "Amount_numeric":
            [10.0, 20.0, 30.0, 40.0],

        "Merchant Name":
            [100, 200, 200, 100],
    }
)


causal_test = (
    compute_causal_behavioral_features(
        causal_test_df
    )
)


np.testing.assert_allclose(
    causal_test[
        "time_since_previous_transaction_min"
    ].to_numpy(),
    np.array(
        [
            np.nan,
            60.0,
            60.0,
            30.0,
        ]
    ),
    equal_nan=True,
)


np.testing.assert_array_equal(
    causal_test[
        "transactions_last_1h"
    ].to_numpy(),
    np.array(
        [
            0,
            1,
            1,
            2,
        ]
    ),
)


np.testing.assert_array_equal(
    causal_test[
        "is_new_merchant"
    ].to_numpy(),
    np.array(
        [
            True,
            True,
            True,
            False,
        ]
    ),
)


print(
    "M6.6 STRICT-CAUSAL REGRESSION GATE: PASS"
)


M6.6 STRICT-CAUSAL REGRESSION GATE: PASS


## 11. Build canonical 10-feature semantic frame

In [9]:

CORE_FEATURE_COLUMNS = [
    "amount_numeric",
    "time_since_previous_transaction_min",
    "transactions_last_1h",
    "amount_minus_previous_mean",
    "is_new_merchant",
    "has_prior_card_history",
    "transaction_mode",
    "location_state",
    "hour_of_day",
    "day_of_week",
]


def build_core_feature_frame(
    context_card_df,
    behavioral_df,
):
    timestamp = (
        context_card_df[
            "Timestamp"
        ]
    )

    frame = pd.DataFrame(
        {
            "raw_row_id":
                context_card_df[
                    "raw_row_id"
                ].to_numpy(),

            "Timestamp":
                timestamp.to_numpy(),

            "amount_numeric":
                behavioral_df[
                    "amount_numeric"
                ].to_numpy(),

            "time_since_previous_transaction_min":
                behavioral_df[
                    "time_since_previous_transaction_min"
                ].to_numpy(),

            "transactions_last_1h":
                behavioral_df[
                    "transactions_last_1h"
                ].to_numpy(),

            "amount_minus_previous_mean":
                behavioral_df[
                    "amount_minus_previous_mean"
                ].to_numpy(),

            "is_new_merchant":
                behavioral_df[
                    "is_new_merchant"
                ].to_numpy(),

            "has_prior_card_history":
                behavioral_df[
                    "has_prior_card_history"
                ].to_numpy(),

            "transaction_mode":
                normalize_string(
                    context_card_df[
                        "Use Chip"
                    ]
                ).to_numpy(),

            "location_state":
                assign_location_state(
                    context_card_df
                ).to_numpy(),

            "hour_of_day":
                (
                    timestamp
                    .dt.hour
                    .astype("Int8")
                    .astype("string")
                    .to_numpy()
                ),

            "day_of_week":
                (
                    timestamp
                    .dt.dayofweek
                    .astype("Int8")
                    .astype("string")
                    .to_numpy()
                ),
        }
    )

    allowed_na = {
        "time_since_previous_transaction_min",
        "amount_minus_previous_mean",
    }

    unexpected_missing = {
        column:
            int(
                frame[
                    column
                ]
                .isna()
                .sum()
            )
        for column
        in CORE_FEATURE_COLUMNS
        if (
            frame[
                column
            ]
            .isna()
            .any()
            and
            column
            not in allowed_na
        )
    }

    if unexpected_missing:
        raise ValueError(
            "Unexpected missing: "
            f"{unexpected_missing}"
        )

    if (
        frame[
            "location_state"
        ]
        .eq(
            "OTHER_INCONSISTENT"
        )
        .any()
    ):
        raise ValueError(
            "OTHER_INCONSISTENT location state."
        )

    return frame


assert len(
    CORE_FEATURE_COLUMNS
) == 10


print(
    "M6.6 CORE SEMANTIC FRAME CONTRACT: LOCKED"
)


M6.6 CORE SEMANTIC FRAME CONTRACT: LOCKED


## 12. Rebuild validation semantic transactions

Pass này có thể mất khoảng vài phút tùy máy vì phải stream full raw artifact theo Card để bảo toàn historical warm-up.

Quan trọng:

- `Is Fraud?` không được đọc;
- behavioral history chỉ nhận rows `< 2019-06-01`;
- chỉ validation rows được giữ lại trong RAM;
- raw User/Card/Merchant Name chỉ tồn tại tạm thời cho causal state và không được persist vào error-analysis artifact.

In [10]:

validation_semantic_parts = []

card_count = 0
raw_rows_seen = 0
context_rows_seen = 0

rebuild_start = (
    time.perf_counter()
)


for card_key, card_df in (
    iter_card_blocks(
        RAW_PATH,
        CHUNK_SIZE,
        SEMANTIC_USECOLS,
    )
):
    card_count += 1
    raw_rows_seen += len(
        card_df
    )

    context_card_df = (
        card_df.loc[
            card_df[
                "Timestamp"
            ]
            < VALIDATION_END
        ]
        .reset_index(
            drop=True
        )
    )

    if context_card_df.empty:
        continue

    context_rows_seen += len(
        context_card_df
    )

    behavior_input = (
        context_card_df[
            [
                "raw_row_id",
                "Timestamp",
                "Amount_numeric",
                "Merchant Name",
            ]
        ]
        .copy()
    )

    assert (
        "Is Fraud?"
        not in behavior_input.columns
    )

    behavioral_df = (
        compute_causal_behavioral_features(
            behavior_input
        )
    )

    semantic_frame = (
        build_core_feature_frame(
            context_card_df,
            behavioral_df,
        )
    )

    validation_mask = (
        (
            semantic_frame[
                "Timestamp"
            ]
            >= TRAIN_END
        )
        &
        (
            semantic_frame[
                "Timestamp"
            ]
            < VALIDATION_END
        )
    )

    if (
        validation_mask.any()
    ):
        validation_semantic_parts.append(
            semantic_frame.loc[
                validation_mask
            ].copy()
        )

    if (
        card_count
        % 500
        == 0
    ):
        print(
            "Card blocks:",
            f"{card_count:,}",
            "| raw rows:",
            f"{raw_rows_seen:,}",
            "| context rows:",
            f"{context_rows_seen:,}",
        )


validation_semantic = pd.concat(
    validation_semantic_parts,
    ignore_index=True,
)


rebuild_elapsed = (
    time.perf_counter()
    - rebuild_start
)


print(
    "\nCard blocks:",
    f"{card_count:,}",
)

print(
    "Raw rows streamed:",
    f"{raw_rows_seen:,}",
)

print(
    "Development/history context rows:",
    f"{context_rows_seen:,}",
)

print(
    "Validation semantic rows:",
    f"{len(validation_semantic):,}",
)

print(
    "Elapsed seconds:",
    round(
        rebuild_elapsed,
        2,
    ),
)


Card blocks: 500 | raw rows: 2,021,584 | context rows: 1,911,651
Card blocks: 1,000 | raw rows: 4,066,794 | context rows: 3,841,967
Card blocks: 1,500 | raw rows: 5,915,634 | context rows: 5,589,029
Card blocks: 2,000 | raw rows: 7,883,446 | context rows: 7,444,960
Card blocks: 2,500 | raw rows: 9,842,479 | context rows: 9,292,973
Card blocks: 3,000 | raw rows: 11,897,259 | context rows: 11,238,345
Card blocks: 3,500 | raw rows: 13,997,374 | context rows: 13,231,256
Card blocks: 4,000 | raw rows: 16,040,304 | context rows: 15,156,787
Card blocks: 4,500 | raw rows: 18,115,490 | context rows: 17,121,544
Card blocks: 5,000 | raw rows: 19,945,815 | context rows: 18,850,177
Card blocks: 5,500 | raw rows: 21,956,418 | context rows: 20,747,229
Card blocks: 6,000 | raw rows: 23,844,247 | context rows: 22,528,539

Card blocks: 6,139
Raw rows streamed: 24,386,900
Development/history context rows: 23,038,920
Validation semantic rows: 712,458
Elapsed seconds: 117.69


## 13. Transaction-level lineage / semantic mapping gate

In [11]:

assert (
    card_count
    == EXPECTED_CARD_COUNT
)

assert (
    raw_rows_seen
    == EXPECTED_RAW_ROWS
)

assert (
    context_rows_seen
    == EXPECTED_CONTEXT_ROWS
)

assert (
    len(
        validation_semantic
    )
    == EXPECTED_VALIDATION_ROWS
)

assert (
    validation_semantic[
        "raw_row_id"
    ]
    .is_unique
)

np.testing.assert_array_equal(
    validation_semantic[
        "raw_row_id"
    ].to_numpy(
        dtype=np.int64
    ),
    row_id_validation,
)


assert (
    validation_semantic[
        "Timestamp"
    ]
    .min()
    >= TRAIN_END
)

assert (
    validation_semantic[
        "Timestamp"
    ]
    .max()
    < VALIDATION_END
)


assert set(
    CORE_FEATURE_COLUMNS
).issubset(
    validation_semantic.columns
)


for prohibited in [
    "User",
    "Card",
    "Merchant Name",
    "Errors?",
    "Is Fraud?",
]:
    assert (
        prohibited
        not in validation_semantic.columns
    )


validation_semantic[
    "is_fraud"
] = y_validation


assert int(
    validation_semantic[
        "is_fraud"
    ].sum()
) == EXPECTED_VALIDATION_FRAUD


print(
    "First validation raw_row_id:",
    int(
        validation_semantic[
            "raw_row_id"
        ].iloc[0]
    ),
)

print(
    "Last validation raw_row_id:",
    int(
        validation_semantic[
            "raw_row_id"
        ].iloc[-1]
    ),
)

print(
    "\nM6.6 TRANSACTION-LEVEL LINEAGE MAPPING GATE: PASS"
)


First validation raw_row_id: 4776
Last validation raw_row_id: 24385729

M6.6 TRANSACTION-LEVEL LINEAGE MAPPING GATE: PASS


## 14. Load six prediction / risk-score artifacts

M6.6 tái kiểm tra:

- persisted array length;
- prediction support;
- risk-score range;
- SHA-256 identity;
- confusion counts.

Không chạy model prediction lại.

In [12]:

registry_by_id = {
    record[
        "experiment_id"
    ]:
    record
    for record
    in m6_02_registry[
        "records"
    ]
}


OFFICIAL_RUNS = [
    "M5-LR-SHORT-B04",
    "M5-LR-LONG-B04",
    "M5-DT-SHORT-B01",
    "M5-DT-LONG-B01",
    "M5-RF-SHORT-B01",
    "M5-RF-LONG-B01",
]


run_arrays = {}


for experiment_id in (
    OFFICIAL_RUNS
):
    record = (
        registry_by_id[
            experiment_id
        ]
    )

    pred_path = (
        PROJECT_ROOT
        / record[
            "prediction_artifact"
        ]
    )

    score_path = (
        PROJECT_ROOT
        / record[
            "risk_score_artifact"
        ]
    )

    assert pred_path.exists()
    assert score_path.exists()

    assert (
        sha256_file(
            pred_path
        )
        == record[
            "prediction_sha256"
        ]
    )

    assert (
        sha256_file(
            score_path
        )
        == record[
            "risk_score_sha256"
        ]
    )

    y_pred = np.load(
        pred_path,
        allow_pickle=False,
    )

    risk_score = np.load(
        score_path,
        allow_pickle=False,
    )

    assert y_pred.ndim == 1
    assert risk_score.ndim == 1

    assert len(
        y_pred
    ) == EXPECTED_VALIDATION_ROWS

    assert len(
        risk_score
    ) == EXPECTED_VALIDATION_ROWS

    assert set(
        np.unique(
            y_pred
        ).tolist()
    ) <= {0, 1}

    assert np.isfinite(
        risk_score
    ).all()

    assert np.all(
        risk_score
        >= 0.0
    )

    assert np.all(
        risk_score
        <= 1.0
    )

    run_arrays[
        experiment_id
    ] = {
        "y_pred":
            y_pred.astype(
                np.int8,
                copy=False,
            ),

        "risk_score":
            risk_score.astype(
                np.float64,
                copy=False,
            ),
    }


assert len(
    run_arrays
) == 6


print(
    "M6.6 SIX-RUN ARRAY IDENTITY GATE: PASS"
)


M6.6 SIX-RUN ARRAY IDENTITY GATE: PASS


## 15. Reconstruct TP / FP / FN / TN row groups

In [13]:

error_group_masks = {}
error_group_row_ids = {}


for experiment_id in (
    OFFICIAL_RUNS
):
    record = (
        registry_by_id[
            experiment_id
        ]
    )

    y_pred = (
        run_arrays[
            experiment_id
        ][
            "y_pred"
        ]
    )

    tp = (
        (y_validation == 1)
        & (y_pred == 1)
    )

    fn = (
        (y_validation == 1)
        & (y_pred == 0)
    )

    fp = (
        (y_validation == 0)
        & (y_pred == 1)
    )

    tn = (
        (y_validation == 0)
        & (y_pred == 0)
    )

    assert int(
        tp.sum()
    ) == int(
        record["tp"]
    )

    assert int(
        fn.sum()
    ) == int(
        record["fn"]
    )

    assert int(
        fp.sum()
    ) == int(
        record["fp"]
    )

    assert int(
        tn.sum()
    ) == int(
        record["tn"]
    )

    assert np.all(
        (
            tp.astype(np.int8)
            + fn.astype(np.int8)
            + fp.astype(np.int8)
            + tn.astype(np.int8)
        )
        == 1
    )

    error_group_masks[
        experiment_id
    ] = {
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn,
    }

    error_group_row_ids[
        experiment_id
    ] = {
        "TP":
            row_id_validation[
                tp
            ],

        "FN":
            row_id_validation[
                fn
            ],

        "FP":
            row_id_validation[
                fp
            ],
    }


print(
    "M6.6 CONFUSION GROUP RECONSTRUCTION GATE: PASS"
)


M6.6 CONFUSION GROUP RECONSTRUCTION GATE: PASS


## 16. Score / baseline decision-rule compatibility

M6.6 không tìm threshold.

Cell này chỉ xác minh rằng persisted risk score nằm đúng phía của reference boundary `0.5` so với persisted `y_pred`.

Nếu không tương thích:

`STOP`

vì score-band interpretation không còn hợp lệ.

In [14]:

REFERENCE_BOUNDARY = 0.5
BOUNDARY_TOL = 1e-12


score_boundary_checks = {}


for experiment_id in (
    OFFICIAL_RUNS
):
    y_pred = (
        run_arrays[
            experiment_id
        ][
            "y_pred"
        ]
    )

    scores = (
        run_arrays[
            experiment_id
        ][
            "risk_score"
        ]
    )

    positive_scores = scores[
        y_pred == 1
    ]

    negative_scores = scores[
        y_pred == 0
    ]

    positive_side_ok = (
        len(
            positive_scores
        )
        == 0
        or
        float(
            positive_scores.min()
        )
        >= (
            REFERENCE_BOUNDARY
            - BOUNDARY_TOL
        )
    )

    negative_side_ok = (
        len(
            negative_scores
        )
        == 0
        or
        float(
            negative_scores.max()
        )
        <= (
            REFERENCE_BOUNDARY
            + BOUNDARY_TOL
        )
    )

    assert positive_side_ok
    assert negative_side_ok

    score_boundary_checks[
        experiment_id
    ] = {
        "reference_boundary":
            REFERENCE_BOUNDARY,

        "positive_min":
            (
                float(
                    positive_scores.min()
                )
                if len(
                    positive_scores
                )
                else None
            ),

        "negative_max":
            (
                float(
                    negative_scores.max()
                )
                if len(
                    negative_scores
                )
                else None
            ),

        "compatible":
            True,
    }


print(
    "M6.6 SCORE / PREDICTION BOUNDARY GATE: PASS"
)


M6.6 SCORE / PREDICTION BOUNDARY GATE: PASS


## 17. Error-analysis summary helpers

In [15]:

NUMERIC_FEATURES = [
    "amount_numeric",
    "time_since_previous_transaction_min",
    "transactions_last_1h",
    "amount_minus_previous_mean",
]

BOOLEAN_FEATURES = [
    "is_new_merchant",
    "has_prior_card_history",
]

CATEGORICAL_FEATURES = [
    "transaction_mode",
    "location_state",
    "hour_of_day",
    "day_of_week",
]


def summarize_numeric(
    series,
):
    values = pd.to_numeric(
        series,
        errors="coerce",
    )

    finite = values[
        values.notna()
    ].to_numpy(
        dtype=np.float64
    )

    result = {
        "n":
            int(
                len(
                    values
                )
            ),

        "missing_count":
            int(
                values.isna().sum()
            ),
    }

    if len(
        finite
    ) == 0:
        result.update(
            {
                "mean": None,
                "std": None,
                "q25": None,
                "median": None,
                "q75": None,
                "q90": None,
                "min": None,
                "max": None,
            }
        )

        return result

    result.update(
        {
            "mean":
                float(
                    np.mean(
                        finite
                    )
                ),

            "std":
                float(
                    np.std(
                        finite
                    )
                ),

            "q25":
                float(
                    np.quantile(
                        finite,
                        0.25,
                    )
                ),

            "median":
                float(
                    np.quantile(
                        finite,
                        0.50,
                    )
                ),

            "q75":
                float(
                    np.quantile(
                        finite,
                        0.75,
                    )
                ),

            "q90":
                float(
                    np.quantile(
                        finite,
                        0.90,
                    )
                ),

            "min":
                float(
                    np.min(
                        finite
                    )
                ),

            "max":
                float(
                    np.max(
                        finite
                    )
                ),
        }
    )

    return result


def summarize_boolean(
    series,
):
    values = (
        series
        .astype("boolean")
    )

    n = len(
        values
    )

    missing_count = int(
        values.isna().sum()
    )

    true_count = int(
        values.fillna(
            False
        ).sum()
    )

    observed_n = (
        n
        - missing_count
    )

    return {
        "n":
            int(
                n
            ),

        "missing_count":
            missing_count,

        "true_count":
            true_count,

        "false_count":
            int(
                observed_n
                - true_count
            ),

        "true_rate":
            (
                float(
                    true_count
                    / observed_n
                )
                if observed_n
                else None
            ),
    }


def categorical_distribution(
    series,
):
    values = (
        series
        .astype("string")
        .fillna(
            "__MISSING__"
        )
    )

    counts = (
        values
        .value_counts(
            dropna=False
        )
    )

    n = len(
        values
    )

    return {
        str(level): {
            "count":
                int(
                    count
                ),

            "rate":
                float(
                    count
                    / n
                ),
        }
        for level, count
        in counts.items()
    }


def categorical_rate_gaps(
    error_distribution,
    correct_distribution,
    top_n=5,
):
    levels = sorted(
        set(
            error_distribution
        )
        |
        set(
            correct_distribution
        )
    )

    rows = []

    for level in levels:
        error_rate = (
            error_distribution
            .get(
                level,
                {
                    "rate":
                        0.0
                },
            )[
                "rate"
            ]
        )

        correct_rate = (
            correct_distribution
            .get(
                level,
                {
                    "rate":
                        0.0
                },
            )[
                "rate"
            ]
        )

        gap = (
            error_rate
            - correct_rate
        )

        rows.append(
            {
                "level":
                    level,

                "error_rate":
                    float(
                        error_rate
                    ),

                "correct_rate":
                    float(
                        correct_rate
                    ),

                "error_minus_correct_rate":
                    float(
                        gap
                    ),

                "absolute_rate_gap":
                    float(
                        abs(
                            gap
                        )
                    ),
            }
        )

    rows.sort(
        key=lambda row:
            row[
                "absolute_rate_gap"
            ],
        reverse=True,
    )

    return rows[
        :top_n
    ]


def analyze_error_vs_correct(
    error_mask,
    correct_mask,
):
    result = {
        "error_count":
            int(
                error_mask.sum()
            ),

        "correct_count":
            int(
                correct_mask.sum()
            ),

        "numeric": {},
        "boolean": {},
        "categorical": {},
    }

    for feature in (
        NUMERIC_FEATURES
    ):
        error_summary = (
            summarize_numeric(
                validation_semantic.loc[
                    error_mask,
                    feature,
                ]
            )
        )

        correct_summary = (
            summarize_numeric(
                validation_semantic.loc[
                    correct_mask,
                    feature,
                ]
            )
        )

        result[
            "numeric"
        ][
            feature
        ] = {
            "error":
                error_summary,

            "correct":
                correct_summary,

            "delta_mean":
                (
                    None
                    if (
                        error_summary[
                            "mean"
                        ]
                        is None
                        or
                        correct_summary[
                            "mean"
                        ]
                        is None
                    )
                    else float(
                        error_summary[
                            "mean"
                        ]
                        - correct_summary[
                            "mean"
                        ]
                    )
                ),

            "delta_median":
                (
                    None
                    if (
                        error_summary[
                            "median"
                        ]
                        is None
                        or
                        correct_summary[
                            "median"
                        ]
                        is None
                    )
                    else float(
                        error_summary[
                            "median"
                        ]
                        - correct_summary[
                            "median"
                        ]
                    )
                ),
        }

    for feature in (
        BOOLEAN_FEATURES
    ):
        error_summary = (
            summarize_boolean(
                validation_semantic.loc[
                    error_mask,
                    feature,
                ]
            )
        )

        correct_summary = (
            summarize_boolean(
                validation_semantic.loc[
                    correct_mask,
                    feature,
                ]
            )
        )

        result[
            "boolean"
        ][
            feature
        ] = {
            "error":
                error_summary,

            "correct":
                correct_summary,

            "delta_true_rate":
                float(
                    error_summary[
                        "true_rate"
                    ]
                    - correct_summary[
                        "true_rate"
                    ]
                ),
        }

    for feature in (
        CATEGORICAL_FEATURES
    ):
        error_distribution = (
            categorical_distribution(
                validation_semantic.loc[
                    error_mask,
                    feature,
                ]
            )
        )

        correct_distribution = (
            categorical_distribution(
                validation_semantic.loc[
                    correct_mask,
                    feature,
                ]
            )
        )

        result[
            "categorical"
        ][
            feature
        ] = {
            "error_distribution":
                error_distribution,

            "correct_distribution":
                correct_distribution,

            "largest_absolute_rate_gaps":
                categorical_rate_gaps(
                    error_distribution,
                    correct_distribution,
                    top_n=5,
                ),
        }

    return result


print(
    "M6.6 ERROR-ANALYSIS HELPERS: DEFINED"
)


M6.6 ERROR-ANALYSIS HELPERS: DEFINED


## 18. Risk-score error context helpers

In [16]:

def summarize_score(
    scores,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    if len(
        scores
    ) == 0:
        return {
            "n": 0,
            "mean": None,
            "q10": None,
            "q25": None,
            "median": None,
            "q75": None,
            "q90": None,
            "min": None,
            "max": None,
        }

    return {
        "n":
            int(
                len(
                    scores
                )
            ),

        "mean":
            float(
                np.mean(
                    scores
                )
            ),

        "q10":
            float(
                np.quantile(
                    scores,
                    0.10,
                )
            ),

        "q25":
            float(
                np.quantile(
                    scores,
                    0.25,
                )
            ),

        "median":
            float(
                np.quantile(
                    scores,
                    0.50,
                )
            ),

        "q75":
            float(
                np.quantile(
                    scores,
                    0.75,
                )
            ),

        "q90":
            float(
                np.quantile(
                    scores,
                    0.90,
                )
            ),

        "min":
            float(
                np.min(
                    scores
                )
            ),

        "max":
            float(
                np.max(
                    scores
                )
            ),
    }


def count_rate(
    mask,
):
    count = int(
        mask.sum()
    )

    n = len(
        mask
    )

    return {
        "count":
            count,

        "rate":
            (
                float(
                    count
                    / n
                )
                if n
                else None
            ),
    }


def build_error_score_context(
    experiment_id,
):
    scores = (
        run_arrays[
            experiment_id
        ][
            "risk_score"
        ]
    )

    masks = (
        error_group_masks[
            experiment_id
        ]
    )

    tp_scores = scores[
        masks["TP"]
    ]

    fn_scores = scores[
        masks["FN"]
    ]

    fp_scores = scores[
        masks["FP"]
    ]

    tn_scores = scores[
        masks["TN"]
    ]

    fn_bands = {
        "very_low_score_lt_0_10":
            count_rate(
                fn_scores
                < 0.10
            ),

        "low_score_0_10_to_lt_0_40":
            count_rate(
                (
                    fn_scores
                    >= 0.10
                )
                &
                (
                    fn_scores
                    < 0.40
                )
            ),

        "near_boundary_0_40_to_0_50":
            count_rate(
                (
                    fn_scores
                    >= 0.40
                )
                &
                (
                    fn_scores
                    <= (
                        REFERENCE_BOUNDARY
                        + BOUNDARY_TOL
                    )
                )
            ),
    }

    fp_bands = {
        "near_boundary_0_50_to_lt_0_60":
            count_rate(
                (
                    fp_scores
                    >= (
                        REFERENCE_BOUNDARY
                        - BOUNDARY_TOL
                    )
                )
                &
                (
                    fp_scores
                    < 0.60
                )
            ),

        "high_score_0_60_to_lt_0_90":
            count_rate(
                (
                    fp_scores
                    >= 0.60
                )
                &
                (
                    fp_scores
                    < 0.90
                )
            ),

        "very_high_score_ge_0_90":
            count_rate(
                fp_scores
                >= 0.90
            ),
    }

    return {
        "reference_boundary":
            REFERENCE_BOUNDARY,

        "threshold_search_performed":
            False,

        "TP":
            summarize_score(
                tp_scores
            ),

        "FN":
            summarize_score(
                fn_scores
            ),

        "FP":
            summarize_score(
                fp_scores
            ),

        "TN":
            summarize_score(
                tn_scores
            ),

        "FN_descriptive_bands":
            fn_bands,

        "FP_descriptive_bands":
            fp_bands,
    }


print(
    "M6.6 SCORE-CONTEXT HELPERS: DEFINED"
)


M6.6 SCORE-CONTEXT HELPERS: DEFINED


## 19. Run six-run transaction-level error analysis

In [17]:

run_error_analysis = {}


for experiment_id in (
    OFFICIAL_RUNS
):
    masks = (
        error_group_masks[
            experiment_id
        ]
    )

    fraud_side = (
        analyze_error_vs_correct(
            error_mask=masks[
                "FN"
            ],
            correct_mask=masks[
                "TP"
            ],
        )
    )

    non_fraud_side = (
        analyze_error_vs_correct(
            error_mask=masks[
                "FP"
            ],
            correct_mask=masks[
                "TN"
            ],
        )
    )

    score_context = (
        build_error_score_context(
            experiment_id
        )
    )

    run_error_analysis[
        experiment_id
    ] = {
        "experiment_id":
            experiment_id,

        "model_family":
            registry_by_id[
                experiment_id
            ][
                "model_family"
            ],

        "training_window_id":
            registry_by_id[
                experiment_id
            ][
                "training_window_id"
            ],

        "confusion_counts": {
            "TP":
                int(
                    masks[
                        "TP"
                    ].sum()
                ),

            "FN":
                int(
                    masks[
                        "FN"
                    ].sum()
                ),

            "FP":
                int(
                    masks[
                        "FP"
                    ].sum()
                ),

            "TN":
                int(
                    masks[
                        "TN"
                    ].sum()
                ),
        },

        "fraud_side_FN_vs_TP":
            fraud_side,

        "non_fraud_side_FP_vs_TN":
            non_fraud_side,

        "risk_score_context":
            score_context,

        "row_level_mapping":
            "VERIFIED",

        "threshold_optimization":
            "NOT_PERFORMED",

        "final_test_accessed":
            False,
    }


assert len(
    run_error_analysis
) == 6


print(
    "M6.6 SIX-RUN TRANSACTION ERROR ANALYSIS GATE: PASS"
)


M6.6 SIX-RUN TRANSACTION ERROR ANALYSIS GATE: PASS


## 20. Concise per-run error evidence

Cell này chỉ in các gap dễ review:

Numeric:

`error median − correct median`

Boolean:

`error true-rate − correct true-rate`

Categorical:

`largest absolute percentage-point gap`

Risk score:

`FN median / FP median + descriptive boundary bands`

Full distributions được persist trong JSON.

In [18]:

for experiment_id in (
    OFFICIAL_RUNS
):
    analysis = (
        run_error_analysis[
            experiment_id
        ]
    )

    print("=" * 72)

    print(
        experiment_id,
        "|",
        analysis[
            "model_family"
        ],
        "|",
        analysis[
            "training_window_id"
        ],
    )

    print(
        "Counts:",
        analysis[
            "confusion_counts"
        ],
    )

    print(
        "\nFraud side — FN vs TP"
    )

    for feature in (
        NUMERIC_FEATURES
    ):
        item = (
            analysis[
                "fraud_side_FN_vs_TP"
            ][
                "numeric"
            ][
                feature
            ]
        )

        print(
            " ",
            feature,
            "Δmedian(FN-TP)=",
            item[
                "delta_median"
            ],
        )

    for feature in (
        BOOLEAN_FEATURES
    ):
        item = (
            analysis[
                "fraud_side_FN_vs_TP"
            ][
                "boolean"
            ][
                feature
            ]
        )

        print(
            " ",
            feature,
            "Δtrue_rate(FN-TP)=",
            item[
                "delta_true_rate"
            ],
        )

    for feature in (
        CATEGORICAL_FEATURES
    ):
        gaps = (
            analysis[
                "fraud_side_FN_vs_TP"
            ][
                "categorical"
            ][
                feature
            ][
                "largest_absolute_rate_gaps"
            ]
        )

        print(
            " ",
            feature,
            "top gap=",
            (
                gaps[0]
                if gaps
                else None
            ),
        )

    print(
        "\nNon-fraud side — FP vs TN"
    )

    for feature in (
        NUMERIC_FEATURES
    ):
        item = (
            analysis[
                "non_fraud_side_FP_vs_TN"
            ][
                "numeric"
            ][
                feature
            ]
        )

        print(
            " ",
            feature,
            "Δmedian(FP-TN)=",
            item[
                "delta_median"
            ],
        )

    for feature in (
        BOOLEAN_FEATURES
    ):
        item = (
            analysis[
                "non_fraud_side_FP_vs_TN"
            ][
                "boolean"
            ][
                feature
            ]
        )

        print(
            " ",
            feature,
            "Δtrue_rate(FP-TN)=",
            item[
                "delta_true_rate"
            ],
        )

    for feature in (
        CATEGORICAL_FEATURES
    ):
        gaps = (
            analysis[
                "non_fraud_side_FP_vs_TN"
            ][
                "categorical"
            ][
                feature
            ][
                "largest_absolute_rate_gaps"
            ]
        )

        print(
            " ",
            feature,
            "top gap=",
            (
                gaps[0]
                if gaps
                else None
            ),
        )

    score_context = (
        analysis[
            "risk_score_context"
        ]
    )

    print(
        "\nFN score median:",
        score_context[
            "FN"
        ][
            "median"
        ],
    )

    print(
        "FP score median:",
        score_context[
            "FP"
        ][
            "median"
        ],
    )

    print(
        "FN bands:",
        score_context[
            "FN_descriptive_bands"
        ],
    )

    print(
        "FP bands:",
        score_context[
            "FP_descriptive_bands"
        ],
    )


print(
    "\nM6.6 PER-RUN ERROR EVIDENCE: READY"
)


M5-LR-SHORT-B04 | Logistic Regression | W_SHORT
Counts: {'TP': 258, 'FN': 794, 'FP': 219, 'TN': 711187}

Fraud side — FN vs TP
  amount_numeric Δmedian(FN-TP)= -7.135000000000005
  time_since_previous_transaction_min Δmedian(FN-TP)= -36.5
  transactions_last_1h Δmedian(FN-TP)= 0.0
  amount_minus_previous_mean Δmedian(FN-TP)= -17.3688664181297
  is_new_merchant Δtrue_rate(FN-TP)= -0.5711049928729033
  has_prior_card_history Δtrue_rate(FN-TP)= 0.0
  transaction_mode top gap= {'level': 'Swipe Transaction', 'error_rate': 0.08564231738035265, 'correct_rate': 0.09689922480620156, 'error_minus_correct_rate': -0.011256907425848908, 'absolute_rate_gap': 0.011256907425848908}
  location_state top gap= {'level': 'PHYSICAL_ZIP_UNAVAILABLE', 'error_rate': 1.0, 'correct_rate': 1.0, 'error_minus_correct_rate': 0.0, 'absolute_rate_gap': 0.0}
  hour_of_day top gap= {'level': '10', 'error_rate': 0.08438287153652393, 'correct_rate': 0.007751937984496124, 'error_minus_correct_rate': 0.07663093355202781, '

## 21. Same-window cross-family error overlap

M6.6 kiểm tra trên cùng training window:

```text
W_SHORT:
LR / DT / RF

W_LONG:
LR / DT / RF
```

Fraud-side:

- all three miss;
- all three catch;
- exactly one model catches;
- exactly two models catch.

Non-fraud-side:

- any false positive;
- all three false positive;
- exactly one model false-positives;
- exactly two models false-positive.

Đây là row-level overlap evidence, không phải model selection.

In [19]:

WINDOW_RUNS = {
    "W_SHORT": [
        "M5-LR-SHORT-B04",
        "M5-DT-SHORT-B01",
        "M5-RF-SHORT-B01",
    ],

    "W_LONG": [
        "M5-LR-LONG-B04",
        "M5-DT-LONG-B01",
        "M5-RF-LONG-B01",
    ],
}


cross_run_overlap = {}
overlap_row_id_arrays = {}


for window_id, experiment_ids in (
    WINDOW_RUNS.items()
):
    prediction_matrix = np.column_stack(
        [
            run_arrays[
                experiment_id
            ][
                "y_pred"
            ]
            for experiment_id
            in experiment_ids
        ]
    )

    assert (
        prediction_matrix.shape
        == (
            EXPECTED_VALIDATION_ROWS,
            3,
        )
    )

    positive_votes = (
        prediction_matrix.sum(
            axis=1
        )
    )

    fraud_rows = (
        y_validation == 1
    )

    non_fraud_rows = (
        y_validation == 0
    )

    fraud_votes = (
        positive_votes[
            fraud_rows
        ]
    )

    non_fraud_votes = (
        positive_votes[
            non_fraud_rows
        ]
    )

    fraud_row_ids = (
        row_id_validation[
            fraud_rows
        ]
    )

    non_fraud_row_ids = (
        row_id_validation[
            non_fraud_rows
        ]
    )

    fraud_summary = {
        "actual_fraud":
            int(
                fraud_rows.sum()
            ),

        "all_three_miss":
            int(
                (
                    fraud_votes
                    == 0
                ).sum()
            ),

        "exactly_one_catches":
            int(
                (
                    fraud_votes
                    == 1
                ).sum()
            ),

        "exactly_two_catch":
            int(
                (
                    fraud_votes
                    == 2
                ).sum()
            ),

        "all_three_catch":
            int(
                (
                    fraud_votes
                    == 3
                ).sum()
            ),

        "caught_by_at_least_one":
            int(
                (
                    fraud_votes
                    >= 1
                ).sum()
            ),
    }

    non_fraud_summary = {
        "actual_non_fraud":
            int(
                non_fraud_rows.sum()
            ),

        "no_model_false_positive":
            int(
                (
                    non_fraud_votes
                    == 0
                ).sum()
            ),

        "exactly_one_false_positive":
            int(
                (
                    non_fraud_votes
                    == 1
                ).sum()
            ),

        "exactly_two_false_positive":
            int(
                (
                    non_fraud_votes
                    == 2
                ).sum()
            ),

        "all_three_false_positive":
            int(
                (
                    non_fraud_votes
                    == 3
                ).sum()
            ),

        "false_positive_by_at_least_one":
            int(
                (
                    non_fraud_votes
                    >= 1
                ).sum()
            ),
    }

    model_specific_catches = {}
    model_specific_false_positives = {}

    fraud_prediction_matrix = (
        prediction_matrix[
            fraud_rows
        ]
    )

    non_fraud_prediction_matrix = (
        prediction_matrix[
            non_fraud_rows
        ]
    )

    for model_index, experiment_id in enumerate(
        experiment_ids
    ):
        other_indices = [
            index
            for index
            in range(3)
            if index
            != model_index
        ]

        only_catch_mask = (
            (
                fraud_prediction_matrix[
                    :,
                    model_index
                ]
                == 1
            )
            &
            (
                fraud_prediction_matrix[
                    :,
                    other_indices
                ]
                .sum(
                    axis=1
                )
                == 0
            )
        )

        only_fp_mask = (
            (
                non_fraud_prediction_matrix[
                    :,
                    model_index
                ]
                == 1
            )
            &
            (
                non_fraud_prediction_matrix[
                    :,
                    other_indices
                ]
                .sum(
                    axis=1
                )
                == 0
            )
        )

        model_specific_catches[
            experiment_id
        ] = int(
            only_catch_mask.sum()
        )

        model_specific_false_positives[
            experiment_id
        ] = int(
            only_fp_mask.sum()
        )

        overlap_row_id_arrays[
            (
                f"{window_id}__"
                f"{experiment_id}__ONLY_CATCH"
            )
        ] = (
            fraud_row_ids[
                only_catch_mask
            ]
        )

        overlap_row_id_arrays[
            (
                f"{window_id}__"
                f"{experiment_id}__ONLY_FP"
            )
        ] = (
            non_fraud_row_ids[
                only_fp_mask
            ]
        )

    cross_run_overlap[
        window_id
    ] = {
        "experiment_ids":
            experiment_ids,

        "fraud_side":
            fraud_summary,

        "non_fraud_side":
            non_fraud_summary,

        "model_specific_only_catches":
            model_specific_catches,

        "model_specific_only_false_positives":
            model_specific_false_positives,

        "final_model_selection":
            False,
    }

    overlap_row_id_arrays[
        (
            f"{window_id}__"
            "ALL_THREE_MISS"
        )
    ] = (
        fraud_row_ids[
            fraud_votes
            == 0
        ]
    )

    overlap_row_id_arrays[
        (
            f"{window_id}__"
            "ALL_THREE_CATCH"
        )
    ] = (
        fraud_row_ids[
            fraud_votes
            == 3
        ]
    )

    overlap_row_id_arrays[
        (
            f"{window_id}__"
            "ALL_THREE_FP"
        )
    ] = (
        non_fraud_row_ids[
            non_fraud_votes
            == 3
        ]
    )


for window_id, summary in (
    cross_run_overlap.items()
):
    print("=" * 72)

    print(
        window_id
    )

    print(
        "Fraud side:",
        summary[
            "fraud_side"
        ],
    )

    print(
        "Non-fraud side:",
        summary[
            "non_fraud_side"
        ],
    )

    print(
        "Model-specific only catches:",
        summary[
            "model_specific_only_catches"
        ],
    )

    print(
        "Model-specific only FPs:",
        summary[
            "model_specific_only_false_positives"
        ],
    )


print(
    "\nM6.6 CROSS-RUN ERROR OVERLAP GATE: PASS"
)


W_SHORT
Fraud side: {'actual_fraud': 1052, 'all_three_miss': 618, 'exactly_one_catches': 127, 'exactly_two_catch': 154, 'all_three_catch': 153, 'caught_by_at_least_one': 434}
Non-fraud side: {'actual_non_fraud': 711406, 'no_model_false_positive': 710602, 'exactly_one_false_positive': 514, 'exactly_two_false_positive': 217, 'all_three_false_positive': 73, 'false_positive_by_at_least_one': 804}
Model-specific only catches: {'M5-LR-SHORT-B04': 31, 'M5-DT-SHORT-B01': 68, 'M5-RF-SHORT-B01': 28}
Model-specific only FPs: {'M5-LR-SHORT-B04': 58, 'M5-DT-SHORT-B01': 388, 'M5-RF-SHORT-B01': 68}
W_LONG
Fraud side: {'actual_fraud': 1052, 'all_three_miss': 810, 'exactly_one_catches': 166, 'exactly_two_catch': 72, 'all_three_catch': 4, 'caught_by_at_least_one': 242}
Non-fraud side: {'actual_non_fraud': 711406, 'no_model_false_positive': 710546, 'exactly_one_false_positive': 802, 'exactly_two_false_positive': 56, 'all_three_false_positive': 2, 'false_positive_by_at_least_one': 860}
Model-specific only

## 22. Persist row-level lineage artifacts

Persist row IDs cho:

- TP;
- FN;
- FP;

của từng official run;

và selected same-window overlap subsets.

Không persist raw User/Card/Merchant Name.

In [20]:

def safe_npz_key(
    text,
):
    return (
        text
        .replace(
            "-",
            "_",
        )
        .replace(
            " ",
            "_",
        )
        .replace(
            "/",
            "_",
        )
    )


ROW_GROUP_PATH = (
    OUTPUT_DIR
    / "m6_06_error_group_row_ids.npz"
)


row_group_npz = {}


for experiment_id, groups in (
    error_group_row_ids.items()
):
    for group_name, row_ids in (
        groups.items()
    ):
        key = safe_npz_key(
            (
                f"{experiment_id}"
                f"__{group_name}"
            )
        )

        row_group_npz[
            key
        ] = np.asarray(
            row_ids,
            dtype=np.int64,
        )


for name, row_ids in (
    overlap_row_id_arrays.items()
):
    row_group_npz[
        safe_npz_key(
            name
        )
    ] = np.asarray(
        row_ids,
        dtype=np.int64,
    )


np.savez_compressed(
    ROW_GROUP_PATH,
    **row_group_npz,
)


assert ROW_GROUP_PATH.exists()
assert ROW_GROUP_PATH.stat().st_size > 0


with np.load(
    ROW_GROUP_PATH,
    allow_pickle=False,
) as artifact:
    persisted_keys = set(
        artifact.files
    )

    assert (
        persisted_keys
        == set(
            row_group_npz
        )
    )

    for key, expected in (
        row_group_npz.items()
    ):
        np.testing.assert_array_equal(
            artifact[key],
            expected,
        )


print(
    "Persisted row-group arrays:",
    len(
        row_group_npz
    ),
)

print(
    "\nM6.6 ROW-GROUP PERSISTENCE / ROUND-TRIP GATE: PASS"
)


Persisted row-group arrays: 36

M6.6 ROW-GROUP PERSISTENCE / ROUND-TRIP GATE: PASS


## 23. Persist error-analysis JSON + manifest

JSON chứa:

- verified lineage;
- semantic contract;
- six-run FN-vs-TP / FP-vs-TN summaries;
- risk-score error context;
- same-window overlap findings;
- source hashes;
- selection boundary.

Full raw transaction identifiers không được persist.

In [21]:

ANALYSIS_VERSION = (
    "M6.6-fp-fn-error-analysis-v1"
)

ANALYSIS_PATH = (
    OUTPUT_DIR
    / "m6_06_error_analysis.json"
)

MANIFEST_PATH = (
    OUTPUT_DIR
    / "m6_06_analysis_manifest.json"
)


analysis_payload = {
    "analysis_version":
        ANALYSIS_VERSION,

    "m6_substep":
        "M6.6",

    "evaluation_population":
        "VALIDATION_2019-01_TO_2019-05",

    "validation_rows":
        EXPECTED_VALIDATION_ROWS,

    "validation_fraud_rows":
        EXPECTED_VALIDATION_FRAUD,

    "lineage": {
        "row_id_validation_sha256":
            m6_02_registry[
                "row_id_validation_sha256"
            ],

        "y_validation_sha256":
            m6_02_registry[
                "y_validation_sha256"
            ],

        "semantic_rebuild_raw_file_size":
            int(
                RAW_PATH.stat().st_size
            ),

        "semantic_rows":
            int(
                len(
                    validation_semantic
                )
            ),

        "exact_row_id_alignment":
            True,

        "transaction_level_mapping":
            "VERIFIED",
    },

    "semantic_feature_columns":
        CORE_FEATURE_COLUMNS,

    "raw_identifiers_persisted":
        False,

    "raw_target_read_for_semantic_rebuild":
        False,

    "strict_causal_history":
        True,

    "development_context_rows":
        int(
            context_rows_seen
        ),

    "semantic_rebuild_elapsed_seconds":
        float(
            rebuild_elapsed
        ),

    "score_reference_boundary":
        REFERENCE_BOUNDARY,

    "score_boundary_checks":
        score_boundary_checks,

    "run_error_analysis":
        run_error_analysis,

    "cross_run_overlap":
        cross_run_overlap,

    "row_group_artifact":
        str(
            ROW_GROUP_PATH.relative_to(
                PROJECT_ROOT
            )
        ),

    "row_group_artifact_sha256":
        sha256_file(
            ROW_GROUP_PATH
        ),

    "threshold_optimization":
        "NOT_PERFORMED",

    "training_window_winner":
        "OPEN",

    "model_family_winner":
        "OPEN",

    "final_model":
        "OPEN",

    "final_threshold":
        "OPEN",

    "final_test_accessed":
        False,
}


analysis_manifest = {
    "m6_substep":
        "M6.6",

    "analysis_version":
        ANALYSIS_VERSION,

    "transaction_level_lineage":
        "PASS",

    "semantic_feature_rebuild":
        "PASS",

    "strict_causal_regression":
        "PASS",

    "official_runs_expected":
        6,

    "official_runs_analyzed":
        len(
            run_error_analysis
        ),

    "confusion_group_reconstruction":
        "PASS",

    "risk_score_boundary_compatibility":
        "PASS",

    "same_window_overlap_groups":
        2,

    "row_group_persistence":
        "PASS",

    "threshold_search":
        "NOT_PERFORMED",

    "model_retraining":
        "NOT_PERFORMED",

    "raw_target_read_for_semantic_rebuild":
        False,

    "final_test_accessed":
        False,

    "decision":
        "OPEN — REQUIRES M6.6 RUNTIME REVIEW",
}


with open(
    ANALYSIS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        analysis_payload,
        file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        analysis_manifest,
        file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


assert ANALYSIS_PATH.exists()
assert MANIFEST_PATH.exists()

assert ANALYSIS_PATH.stat().st_size > 0
assert MANIFEST_PATH.stat().st_size > 0


print("Analysis artifact:")
print(ANALYSIS_PATH)

print("\nAnalysis manifest:")
print(MANIFEST_PATH)

print(
    "\nM6.6 ANALYSIS PERSISTENCE GATE: PASS"
)


Analysis artifact:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m6_06_fp_fn_error_analysis/m6_06_error_analysis.json

Analysis manifest:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m6_06_fp_fn_error_analysis/m6_06_analysis_manifest.json

M6.6 ANALYSIS PERSISTENCE GATE: PASS


## 24. Persisted JSON round-trip

In [22]:

with open(
    ANALYSIS_PATH,
    "r",
    encoding="utf-8",
) as file:
    analysis_roundtrip = json.load(
        file
    )


with open(
    MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    manifest_roundtrip = json.load(
        file
    )


assert (
    analysis_roundtrip[
        "analysis_version"
    ]
    == ANALYSIS_VERSION
)

assert (
    analysis_roundtrip[
        "lineage"
    ][
        "transaction_level_mapping"
    ]
    == "VERIFIED"
)

assert (
    analysis_roundtrip[
        "lineage"
    ][
        "exact_row_id_alignment"
    ]
    is True
)

assert (
    len(
        analysis_roundtrip[
            "run_error_analysis"
        ]
    )
    == 6
)

assert set(
    analysis_roundtrip[
        "cross_run_overlap"
    ]
) == {
    "W_SHORT",
    "W_LONG",
}

assert (
    analysis_roundtrip[
        "raw_identifiers_persisted"
    ]
    is False
)

assert (
    analysis_roundtrip[
        "raw_target_read_for_semantic_rebuild"
    ]
    is False
)

assert (
    analysis_roundtrip[
        "threshold_optimization"
    ]
    == "NOT_PERFORMED"
)

assert (
    analysis_roundtrip[
        "final_test_accessed"
    ]
    is False
)


assert (
    manifest_roundtrip[
        "transaction_level_lineage"
    ]
    == "PASS"
)

assert (
    manifest_roundtrip[
        "official_runs_analyzed"
    ]
    == 6
)

assert (
    manifest_roundtrip[
        "row_group_persistence"
    ]
    == "PASS"
)

assert (
    manifest_roundtrip[
        "final_test_accessed"
    ]
    is False
)


print(
    "M6.6 ANALYSIS ROUND-TRIP GATE: PASS"
)


M6.6 ANALYSIS ROUND-TRIP GATE: PASS


## 25. FINAL TEST / leakage isolation gate

Required:

```text
raw Is Fraud?:
NOT READ

raw Errors?:
NOT READ

semantic/history max timestamp:
< 2019-06-01

validation semantic frame:
2019-01-01 <= Timestamp < 2019-06-01

threshold optimization:
NONE

FINAL TEST score:
NOT COMPUTED
```

In [23]:

assert (
    "Is Fraud?"
    not in SEMANTIC_USECOLS
)

assert (
    "Errors?"
    not in SEMANTIC_USECOLS
)

assert (
    validation_semantic[
        "Timestamp"
    ]
    .min()
    >= TRAIN_END
)

assert (
    validation_semantic[
        "Timestamp"
    ]
    .max()
    < VALIDATION_END
)

assert (
    context_rows_seen
    == EXPECTED_CONTEXT_ROWS
)

assert (
    analysis_payload[
        "raw_target_read_for_semantic_rebuild"
    ]
    is False
)

assert (
    analysis_payload[
        "threshold_optimization"
    ]
    == "NOT_PERFORMED"
)

assert (
    analysis_payload[
        "final_test_accessed"
    ]
    is False
)

assert (
    m6_02_registry[
        "final_test_accessed"
    ]
    is False
)

assert (
    m6_05_comparison[
        "final_test_accessed"
    ]
    is False
)


print(
    "M6.6 FINAL TEST / LEAKAGE ISOLATION GATE: PASS"
)


M6.6 FINAL TEST / LEAKAGE ISOLATION GATE: PASS


## 26. M6.6 overall technical gate

In [24]:

m6_06_gates = {
    "G01_ENVIRONMENT_RAW_ARTIFACT":
        True,

    "G02_UPSTREAM_HANDOFF":
        True,

    "G03_CANONICAL_VALIDATION_LINEAGE":
        True,

    "G04_DETERMINISTIC_REPRESENTATION":
        True,

    "G05_CARD_BLOCK_STREAMING":
        True,

    "G06_STRICT_CAUSAL_REGRESSION":
        True,

    "G07_CORE_SEMANTIC_CONTRACT":
        True,

    "G08_TRANSACTION_LEVEL_MAPPING":
        True,

    "G09_SIX_RUN_ARRAY_IDENTITY":
        True,

    "G10_CONFUSION_GROUP_RECONSTRUCTION":
        True,

    "G11_SCORE_PREDICTION_BOUNDARY":
        True,

    "G12_SIX_RUN_ERROR_ANALYSIS":
        True,

    "G13_PER_RUN_ERROR_EVIDENCE":
        True,

    "G14_CROSS_RUN_ERROR_OVERLAP":
        True,

    "G15_ROW_GROUP_PERSISTENCE":
        True,

    "G16_ANALYSIS_PERSISTENCE":
        True,

    "G17_ANALYSIS_ROUND_TRIP":
        True,

    "G18_FINAL_TEST_LEAKAGE_ISOLATION":
        True,
}


for gate_name, gate_value in (
    m6_06_gates.items()
):
    print(
        gate_name,
        "→",
        (
            "PASS"
            if gate_value
            else "FAIL"
        ),
    )


assert all(
    m6_06_gates.values()
)


print(
    "\nM6.6 OVERALL TECHNICAL GATE: PASS"
)


G01_ENVIRONMENT_RAW_ARTIFACT → PASS
G02_UPSTREAM_HANDOFF → PASS
G03_CANONICAL_VALIDATION_LINEAGE → PASS
G04_DETERMINISTIC_REPRESENTATION → PASS
G05_CARD_BLOCK_STREAMING → PASS
G06_STRICT_CAUSAL_REGRESSION → PASS
G07_CORE_SEMANTIC_CONTRACT → PASS
G08_TRANSACTION_LEVEL_MAPPING → PASS
G09_SIX_RUN_ARRAY_IDENTITY → PASS
G10_CONFUSION_GROUP_RECONSTRUCTION → PASS
G11_SCORE_PREDICTION_BOUNDARY → PASS
G12_SIX_RUN_ERROR_ANALYSIS → PASS
G13_PER_RUN_ERROR_EVIDENCE → PASS
G14_CROSS_RUN_ERROR_OVERLAP → PASS
G15_ROW_GROUP_PERSISTENCE → PASS
G16_ANALYSIS_PERSISTENCE → PASS
G17_ANALYSIS_ROUND_TRIP → PASS
G18_FINAL_TEST_LEAKAGE_ISOLATION → PASS

M6.6 OVERALL TECHNICAL GATE: PASS


# 27. Runtime review registry

Sau Run All, phải review output thật trước khi khóa M6.6.

## M6.6-F01 — Execution integrity

Cần xác minh:

- toàn bộ code cells executed;
- không exception;
- không stderr bất thường.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F02 — Transaction-level lineage

Cần xác minh:

```text
reconstructed validation raw_row_id
==
M4.7 row_id_validation
```

exact equality.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F03 — Semantic feature reconstruction

Cần xác minh:

```text
712,458 validation rows
10 canonical semantic features
strict causal history
```

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F04 — FN vs TP findings

Cần review cho 6 runs:

- Amount;
- recency;
- last-1h activity;
- historical amount deviation;
- new merchant;
- prior-card history;
- transaction mode;
- location state;
- hour;
- day of week.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F05 — FP vs TN findings

Cùng semantic dimensions.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F06 — Risk-score context

Cần review:

- FN score distribution;
- FP score distribution;
- descriptive boundary bands.

Không threshold optimization.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F07 — Shared fraud misses

Cần review:

```text
all-three-model miss
exactly-one catches
exactly-two catch
all-three catch
```

riêng W_SHORT và W_LONG.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F08 — Shared false positives

Cần review:

- any-model FP;
- all-three FP;
- exactly-one/two model FP;
- model-specific FP subsets.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F09 — Error-row lineage persistence

Cần xác minh NPZ row-group artifact round-trip.

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F10 — Leakage / FINAL TEST guardrail

Cần xác minh:

```text
raw target not read
history stops before 2019-06-01
FINAL TEST not evaluated
```

Status:

`OPEN — PENDING RUNTIME REVIEW`

---

## M6.6-F11 — M6.7 handoff

Chỉ READY sau runtime review.

Status:

`OPEN — PENDING RUNTIME REVIEW`

# 28. Decision Log — trước runtime review

## M6.6-D01 — Canonical lineage anchor

Decision:

`M4.7 row_id_validation.npy`

Status:

`INHERITED — LOCKED`

---

## M6.6-D02 — Semantic reconstruction source

Decision:

Use canonical raw IBM TabFormer artifact and exact M4.7 deterministic/causal semantics.

Status:

`LOCKED`

---

## M6.6-D03 — Raw target use

Decision:

Do not read raw `Is Fraud?` in semantic reconstruction.

Label comes from aligned canonical `y_validation`.

Status:

`LOCKED`

---

## M6.6-D04 — Historical rule

Decision:

`Timestamp(history) < Timestamp(current)`

Same timestamp:

`NOT HISTORY`

Status:

`INHERITED — LOCKED`

---

## M6.6-D05 — History boundary

Decision:

History/semantic state stops before:

`2019-06-01`

Status:

`LOCKED`

---

## M6.6-D06 — Fraud-side groups

Decision:

`FN vs TP`

Status:

`LOCKED`

---

## M6.6-D07 — Non-fraud-side groups

Decision:

`FP vs TN`

Status:

`LOCKED`

---

## M6.6-D08 — Semantic dimensions

Decision:

Use 10 canonical pre-encoding semantic features only.

Status:

`LOCKED`

---

## M6.6-D09 — Raw identifiers

Decision:

User/Card/Merchant Name may be temporary causal state keys but are not persisted in M6.6 findings.

Status:

`LOCKED`

---

## M6.6-D10 — Risk score

Decision:

Descriptive error context only.

Reference boundary:

`0.5`

only after prediction/score compatibility gate.

Status:

`LOCKED`

---

## M6.6-D11 — Threshold search

Decision:

None.

Status:

`LOCKED`

---

## M6.6-D12 — Cross-run overlap

Decision:

Analyze LR/DT/RF overlap separately inside W_SHORT and W_LONG.

Status:

`LOCKED`

---

## M6.6-D13 — Row-level lineage artifact

Decision:

Persist TP/FN/FP and selected overlap row IDs.

Status:

`LOCKED`

---

## M6.6-D14 — Final model / window / threshold

Decision:

Remain OPEN.

Status:

`INHERITED — LOCKED BOUNDARY`

---

## M6.6-D15 — FINAL TEST

Decision:

No evaluation/access for M6 decision.

Status:

`INHERITED — LOCKED`

---

## M6.6-D16 — M6.7 handoff

Status:

`OPEN — REQUIRES RUNTIME REVIEW`

# 29. Kết luận trước runtime

Notebook M6.6 đã được thiết kế để:

- verify raw artifact identity;
- verify M4.7 target/row-id lineage;
- reproduce exact raw-row ID semantics;
- rebuild strict-causal transaction-level semantic features;
- prove exact mapping to 712,458 validation rows;
- reconstruct TP/FN/FP/TN for all 6 official runs;
- analyze FN vs TP;
- analyze FP vs TN;
- add descriptive risk-score context;
- inspect same-window shared/model-specific error subsets;
- persist row-level error lineage;
- keep raw identifiers out of findings;
- avoid raw target use during semantic rebuild;
- avoid threshold optimization;
- protect FINAL TEST.

Trước Run All:

```text
M6.6:
NOT YET VERIFIED

Transaction-level mapping:
PENDING

Semantic validation rows:
EXPECTED 712,458

Official runs:
6

Primary error groups:
FN vs TP
FP vs TN

Probability context:
DESCRIPTIVE ONLY

Threshold search:
NONE

Training-window winner:
OPEN

Model-family winner:
OPEN

Final model:
OPEN

Final threshold:
OPEN

FINAL TEST:
PROTECTED

Next:
Run All → runtime review
```

Chỉ sau runtime review mới được khóa:

`M6.6 — PASS`

và:

`READY FOR M6.7`